# 🔬 GIADA Task 1b — Perché il gate `m` manca `1e-3`?

Un singolo run fattoriale separa densità dei dati, capacità, obiettivo e budget. La selezione usa soltanto development; una nuova conferma disgiunta viene aperta dopo il freeze.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
WORKSPACE=Path('/kaggle/working/giada_task_1b');GIADA_REPO=WORKSPACE/'giada';TEACHER_REPO=WORKSPACE/'neuron_as_deep_net'
OUTPUT_DIR=Path('/kaggle/working/artifacts/giada_gate_m_data_budget_capacity_diagnosis')
def run(args,cwd=None):
 print('+',' '.join(map(str,args)));subprocess.run(list(map(str,args)),cwd=cwd,check=True)

In [ ]:
WORKSPACE.mkdir(parents=True,exist_ok=True)
if not GIADA_REPO.exists():run(['git','clone','--branch','codex/surrogate-validity-audit','--single-branch','https://github.com/Zagred47/giada.git',GIADA_REPO])
else:run(['git','fetch','origin','codex/surrogate-validity-audit'],cwd=GIADA_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=GIADA_REPO)
if not TEACHER_REPO.exists():run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',TEACHER_REPO])
run(['git','checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],cwd=TEACHER_REPO)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=GIADA_REPO,text=True).strip();print({'giada_revision':REVISION})

In [ ]:
sys.path.insert(0,str(GIADA_REPO));import torch
assert torch.cuda.is_available(),'Task 1b richiede una GPU CUDA.'
from src.giada_teacher import ExtractedGateFormula,GateMDiagnosisConfig,prepare_gate_m_diagnosis,run_gate_m_diagnosis,evaluate_gate_m_diagnosis
prereg=json.loads((GIADA_REPO/'experiments/teacher_gate_m_diagnostic_preregistration_v1.json').read_text())
display({'gpu':torch.cuda.get_device_name(0),'factorial_axes':prereg['factorial_axes'],'decisions':prereg['decisions']})

In [ ]:
assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
formula=ExtractedGateFormula.from_mod(TEACHER_REPO/'L5PC_NEURON_simulation/mods/Ca_HVA.mod')
bundle=prepare_gate_m_diagnosis(formula);config=GateMDiagnosisConfig()
display(bundle['contract']);assert not bundle['contract']['old_task1_sealed_rows_reused']

In [ ]:
training=run_gate_m_diagnosis(bundle,OUTPUT_DIR,config)
display({'valid':training['valid'],'winner_on_development':training['winner'],'fresh_confirmation_accessed':training['fresh_confirmation_accessed'],'selection':training['selection']})
assert training['valid'] and not training['fresh_confirmation_accessed']

In [ ]:
report=evaluate_gate_m_diagnosis(bundle,OUTPUT_DIR,config)
display({'valid':report['valid'],'winner':report['winner'],'winner_fresh_rmse':report['fresh_confirmation_macro_rmse'][report['winner']],'winner_rollout':report['winner_rollout_rmse'],'diagnosis':report['diagnosis'],'scientific_gate':report['scientific_gate_passed'],'engineering_continuation_gate':report['engineering_continuation_gate_passed']})
display({'factor_effects':report['factor_effects'],'budget_probes':report['budget_probes']})
assert report['valid'] and not report['old_task1_sealed_rows_reused'] and not report['fresh_confirmation_used_for_selection']

## 📦 Download robusto Blob/base64

In [ ]:
from base64 import b64encode
from IPython.display import Javascript,display
archive=Path(shutil.make_archive('/kaggle/working/giada_gate_m_data_budget_capacity_diagnosis','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name))
payload=b64encode(archive.read_bytes()).decode('ascii');filename=archive.name
display(Javascript(f"""const raw=atob('{payload}');const bytes=new Uint8Array(raw.length);for(let i=0;i<raw.length;i++)bytes[i]=raw.charCodeAt(i);const url=URL.createObjectURL(new Blob([bytes],{{type:'application/zip'}}));const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})